<a href="https://colab.research.google.com/github/v35001/aoid-team2-ahiyp/blob/main/code/05_table1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd

# 1. 데이터 불러오기 및 가중치/층화 변수 설정
DATA = 'knhanes_like_week4.csv'
df = pd.read_csv(DATA)
w  = df['wt'].to_numpy()
ST, PS = df['kstrata'].to_numpy(), df['psu'].to_numpy()

# 2. 필수 함수 정의 (도메인 분석용)
def _var_psu(z, strata, psu):
    t = pd.DataFrame({'st': strata, 'cl': psu, 'z': z})
    zc = t.groupby(['st', 'cl'])['z'].sum().reset_index()
    var = 0.0
    for s, g in zc.groupby('st'):
        m = len(g)
        if m > 1:
            var += m / (m - 1) * np.sum((g['z'] - g['z'].mean()) ** 2)
    return var

def svymean_domain(y, w, strata, psu, domain):
    y = np.asarray(y, float)
    d = np.asarray(domain, float) * (~np.isnan(y))
    y = np.nan_to_num(y) * d
    wd = np.asarray(w, float) * d
    th = np.sum(wd * y) / np.sum(wd)
    return th, np.sqrt(_var_psu(wd * (y - th) / np.sum(wd), strata, psu))

# 3. 고위험음주 노출군 / 비노출군 도메인 생성
exp_yes = (df['drink'] == 3).to_numpy() # 노출군 (고위험음주)
exp_no  = (df['drink'] != 3).to_numpy() # 비노출군 (비고위험음주)

DOMS = {
    '전체(n=5,367)': np.ones(len(df), bool),
    '노출군 (고위험음주, n=1,318)': exp_yes,
    '비노출군 (비고위험음주, n=4,049)': exp_no
}

# 4. Table 1 행 생성 함수
def row_cont(label, var):
    r = {'특성': label, 'n': int(df[var].notna().sum())}
    for name, dom in DOMS.items():
        th, se = svymean_domain(df[var], w, ST, PS, dom)
        r[name] = f'{th:.1f} ({se:.2f})'
    return r

def row_cat(label, mask):
    mask = np.asarray(mask, float)
    r = {'특성': label, 'n': int(np.nan_to_num(mask).sum())}
    for name, dom in DOMS.items():
        th, se = svymean_domain(mask, w, ST, PS, dom)
        r[name] = f'{th*100:.1f}'
    return r

# 5. Table 1 조립 (요청 변수 제외 반영)
t1_team = pd.DataFrame([
    row_cont('연령 (세)', 'age'),
    row_cat('  65세 이상', df.age >= 65),
    row_cont('BMI (kg/m²)', 'bmi'),
    row_cont('수축기혈압 (mmHg)', 'sbp'),
    row_cat('고혈압', df.htn == 1),
    row_cat('현재흡연', df.smk == 1),
    row_cont('공복혈당 (mg/dL)', 'glucose'),
    row_cont('총콜레스테롤 (mg/dL)', 'chol'),
    row_cat('사망', df.death == 1),
])

t1_team

,특성,n,"전체(n=5,367)","노출군 (고위험음주, n=1,318)","비노출군 (비고위험음주, n=4,049)"
0,연령 (세),5367,49.1 (0.16),48.6 (0.39),49.3 (0.20)
1,65세 이상,1607,16.8,16.6,16.9
2,BMI (kg/m²),5367,23.8 (0.05),23.7 (0.10),23.9 (0.06)
3,수축기혈압 (mmHg),5367,122.8 (0.45),124.3 (0.67),122.3 (0.45)
4,고혈압,1652,24.4,26.2,23.8
5,현재흡연,1295,24.1,24.2,24.1
6,공복혈당 (mg/dL),5000,91.6 (0.25),92.2 (0.50),91.4 (0.28)
7,총콜레스테롤 (mg/dL),4806,170.4 (0.49),170.6 (0.93),170.3 (0.60)
8,사망,451,5.2,4.7,5.4
